# L19: Agent架构与工具调用


# L19: Agent架构与工具调用

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 解释Agent的核心概念：感知→规划→执行→反馈的循环
2. 理解ReAct、Chain-of-Thought等Agent推理框架
3. 掌握Tool（工具）调用的原理与实现方法
4. 使用LangChain等框架构建能够调用外部工具的Agent
5. 设计多步骤的Agent任务流程，实现复杂任务自动化

## 2. 核心知识点

### 2.1 Agent概述

**什么是Agent**: 能够自主感知环境、做出决策、执行动作的智能系统。Agent不仅能回答问题，还能执行具体的操作。

**Agent核心组件**:


In [ ]:
┌─────────────────────────────────────────────────────────┐
│                      Agent                              │
│  ┌─────────┐    ┌─────────┐    ┌─────────┐    ┌───────┐ │
│  │ 感知   │ -> │ 规划    │ -> │ 执行    │ -> │ 反馈  │ │
│  │Perceive│    │Plan     │    │Act      │    │Observe│ │
│  └─────────┘    └─────────┘    └─────────┘    └───────┘ │
│       ^                                      │          │
│       └──────────────────────────────────────┘          │
│                  观察/反馈循环                          │
└─────────────────────────────────────────────────────────┘
        ↑                              ↑
    工具调用                        环境反馈


**ReAct (Reasoning + Acting)**:
- 核心思想：在执行动作时同时进行推理，推理指导动作选择
- 循环：思考→行动→观察→重复


In [ ]:
问题：今天北京天气如何？适合穿什么衣服？

思考：我需要先查询北京的天气，然后根据天气建议穿衣。
行动：调用天气API获取北京实时天气
观察：天气API返回：今天北京晴转多云，15-25°C，微风
思考：根据天气温度，建议穿轻薄长袖，早晚可加薄外套。
行动：生成穿衣建议
最终回答：今天北京15-25°C，天气晴朗，建议穿轻薄长袖...


### 2.2 工具调用（Tool Use）

**为什么Agent需要工具**:
- LLM本身无法访问实时信息（天气、股价、新闻）
- LLM无法执行具体操作（发邮件、写文件、搜索网页）
- 工具扩展了LLM的能力边界

**常用工具类型**:
| 工具 | 功能 | 示例 |
|------|------|------|
| 搜索 | 网络信息检索 | Google Search, Bing |
| 数据库 | 结构化数据查询 | SQL执行, Knowledge Graph |
| API | 第三方服务调用 | 天气API, 地图API |
| 文件 | 读写本地文件 | Read, Write, Append |
| 代码执行 | 运行代码 | Python REPL, Jupyter |
| 计算器 | 精确数学计算 | Calculator |

**工具调用格式**:


In [ ]:
# OpenAI Function Calling格式
{
    "name": "get_weather",
    "description": "获取指定城市的天气信息",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "城市名称，如北京、上海"
            }
        },
        "required": ["city"]
    }
}

# LLM返回的工具调用
{
    "tool_calls": [{
        "id": "call_123",
        "name": "get_weather",
        "arguments": {"city": "北京"}
    }]
}


### 2.3 LangChain Agent实现


In [ ]:
"""
L19-langchain-agent.py
使用LangChain构建Tool-Calling Agent
"""

from langchain.agents import AgentType, initialize_agent
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool, WikipediaQueryRun, WikipediaQueryRun
from langchain.utilities import WikipediaAPIWrapper

# 初始化LLM
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

# 定义工具
def search_google(query: str) -> str:
    """搜索Google获取实时信息"""
    # 实际使用需要配置Google API
    return f"搜索结果：{query}相关的最新信息..."

search_tool = Tool(
    name="Google搜索",
    func=search_google,
    description="当你需要查询实时信息、新闻或不确定的知识时使用。输入应该是搜索关键词。"
)

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
wiki_tool = Tool(
    name="Wikipedia",
    func=wikipedia.run,
    description="搜索Wikipedia获取百科知识。适合查询事实性信息。"
)

def calculator(expression: str) -> str:
    """计算数学表达式"""
    try:
        result = eval(expression)
        return f"计算结果：{result}"
    except Exception as e:
        return f"计算错误：{str(e)}"

calc_tool = Tool(
    name="计算器",
    func=calculator,
    description="执行数学计算。输入应该是有效的Python数学表达式，如2+2*3"
)

# 构建Agent
tools = [search_tool, wiki_tool, calc_tool]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CHAT_CONVERSATIONAL,
    verbose=True
)

# 测试Agent
response = agent.run("北京今天的天气怎么样？适合户外运动吗？")
print(response)


### 2.4 Chain-of-Thought与ReAct实现


In [ ]:
"""
L19-react-agent.py
实现ReAct推理循环：思考→行动→观察→重复
"""

class ReActAgent:
    def __init__(self, llm, tools):
        self.llm = llm
        self.tools = {t.name: t for t in tools}
        self.max_iterations = 10
        
    def think_and_act(self, question):
        """
        ReAct循环：推理+行动
        """
        history = []
        
        for i in range(self.max_iterations):
            print(f"\n{'='*50}")
            print(f"步骤 {i+1}")
            
            # 1. 思考：分析当前状态，决定下一步行动
            prompt = f"""
问题：{question}

当前历史信息：
{chr(10).join(history)}

请分析：
1. 目前已经知道什么？
2. 还需要什么信息？
3. 下一步应该采取什么行动？（选择一个工具或给出最终答案）

如果需要调用工具，请按以下格式回复：
行动：工具名称
参数：{{"参数名": "参数值"}}

如果已经有足够信息回答问题，请回复：
最终答案：[你的完整回答]
"""
            
            response = self.llm.predict(prompt)
            print(f"思考：{response}")
            
            # 解析响应
            if "最终答案" in response:
                final_answer = response.split("最终答案：")[1].strip()
                return final_answer
            
            if "行动：" in response:
                action_line = response.split("行动：")[1].split("\n")[0].strip()
                params_line = response.split("参数：")[1].strip() if "参数：" in response else "{}"
                params = eval(params_line)  # 简化处理
                
                # 执行工具
                tool_name = action_line
                if tool_name in self.tools:
                    print(f"执行工具：{tool_name}，参数：{params}")
                    result = self.tools[tool_name].run(params)
                    print(f"观察结果：{result}")
                    history.append(f"行动：调用{tool_name}")
                    history.append(f"结果：{result}")
                else:
                    history.append(f"错误：未知工具 {tool_name}")
        
        return "抱歉，我无法在有限步骤内回答这个问题。"

# 使用示例
from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# 假设有一些工具定义
# tools = [search_tool, wiki_tool, calc_tool]
# agent = ReActAgent(llm, tools)
# answer = agent.think_and_act("北京的人口有多少？与上海相比如何？")


### 2.5 自主Agent案例：AutoGPT/HuggingGPT

**AutoGPT**:
- 自主设定目标→分解任务→执行→评估→调整
- 每次行动后评估是否接近目标
- 可调用多种工具：搜索、文件操作、代码执行

**HuggingGPT**:
- 利用HuggingFace上众多专用模型
- 根据任务需求自动选择和组合模型
- LLM作为协调器（Coordinator）

**Agent架构对比**:
| 系统 | 核心思想 | 工具调用方式 | 适用场景 |
|------|----------|--------------|----------|
| AutoGPT | 自主目标分解 | 循环调用多种工具 | 复杂开放式任务 |
| HuggingGPT | 模型编排 | LLM调度专用模型 | 多模态任务 |
| ChatGPT Plugins | 插件生态 | 预定义工具接口 | 垂直场景扩展 |
| Claude Tool Use | 函数调用 | 结构化工具定义 | 精准控制 |

## 3. 代码示例


In [ ]:
"""
L19-practical-agent.py
实战：构建一个能搜索、计算、写作的实用Agent
"""

from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationBufferMemory

# 初始化LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

# 加载预定义工具
tools = load_tools(["serpapi", "python", "llm-math"], llm=llm)

# 添加自定义工具
def write_to_file(filename: str, content: str) -> str:
    """写入文件"""
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(content)
    return f"文件 {filename} 已保存成功！"

def read_from_file(filename: str) -> str:
    """读取文件"""
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return f"文件 {filename} 不存在"

from langchain.tools import Tool
file_tools = [
    Tool(name="写文件", func=write_to_file, description="将内容写入文件，参数：filename(文件名), content(内容)"),
    Tool(name="读文件", func=read_from_file, description="读取文件内容，参数：filename(文件名)")
]

all_tools = tools + file_tools

# 构建带记忆的Agent
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

agent = initialize_agent(
    all_tools,
    llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

# ============ 使用示例 ============
print("=" * 60)
print("实用Agent演示：搜索 → 计算 → 写作")
print("=" * 60)

# 任务1：搜索最新信息
task1 = """
请帮我完成以下任务：
1. 搜索"2024年最受欢迎的编程语言"
2. 列出前5名及其市场份额百分比
3. 计算它们的市场份额总和
4. 将结果保存到文件 programming_languages.txt
"""

print(f"\n任务：{task1}")
# response = agent.run(task1)
print("\n提示：实际运行时需要配置SERPAPI密钥")
print("Agent可以自主：搜索网络 → 计算数据 → 写入文件")

# 任务2：多轮对话
task2 = """
用户说："我想了解人工智能在医疗领域的应用"
请搜索相关信息，然后撰写一份简短的介绍。
"""

print(f"\n任务2：{task2}")
# response = agent.run(task2)


## 4. 练习题

1. **概念理解**: 解释Agent的"感知→规划→执行→反馈"循环，并用生活中的一个例子说明。
2. **ReAct实现**: 不使用LangChain，自己实现一个简化版的ReAct Agent，能够处理至少3种不同类型的工具调用。
3. **工具设计**: 为以下场景设计工具接口（名称、描述、参数），并说明LLM如何决定调用哪个工具：
   - 查询数据库获取销售数据
   - 发送邮件给指定收件人
   - 预订会议室
4. **Agent对比实验**: 使用LangChain或类似框架，分别实现一个ReAct Agent和一个Chain-of-Thought Agent，在至少5个不同任务上对比它们的性能差异。
5. **自主Agent设计**: 设计一个能够自动完成"研究报告撰写"的Agent系统，包括：目标设定、任务分解、信息搜集、数据分析、报告生成、格式优化等步骤，并评估其局限性和潜在风险。

## 5. 延伸阅读

- **论文**: Yao et al., 2022, "ReAct: Synergizing Reasoning and Acting in Language Models" — ReAct原始论文
- **论文**: Shinn et al., 2023, "HuggingGPT: Solving AI Tasks with ChatGPT and its Friends in HuggingFace" — HuggingGPT论文
- **GitHub**: LangChain Agents文档 — https://python.langchain.com/docs/modules/agents/
- **GitHub**: AutoGPT — https://github.com/Significant-Gravitas/AutoGPT
- **网站**: Tool Learning Survey — https://arxiv.org/abs/2304.08354
- **工具**: SerpAPI — Google搜索API，用于获取实时信息

---
